In [1]:
using DifferentialEquations # for the actual time evolution
using OrdinaryDiffEq # for ODEs
using Plots # for plotting
using Base.Threads # for parallelization
using StaticArrays # somehow needed to use multiple variables in DifferentialEquations.jl

using Plots, LaTeXStrings, Colors
using Plots.PlotMeasures
using LinearAlgebra

using Random, Distributions

using FFTW # discrete Fourier transform

using JLD2 # for file saving

In [2]:
level = "../../../../"

include(joinpath(level, "src/4th-order-FD-stencils.jl"));
include(joinpath(level, "src/evolution_Liouville_larger_cutoff.jl"));
include(joinpath(level, "src/hamiltonian_Liouville.jl"));
include(joinpath(level, "src/initial_data_waves.jl"));
include(joinpath(level, "src/visualisation.jl"));

### evolution

In [3]:
function artisan_evolution_at_resolution(Nx, stableRandomSeed, pModel, pInit, target_time)
    # unpack model parameters
    (mphi2, mchi2, c4, c, epsDiss) = pModel;
    
    # set the spatial discretization
    NboundaryPadding = 2;#Int(div(Nx,2));  # Number of boundary padding points
    dx = 1/(Nx);  # Grid spacing
        
    pGrid = (dx, Nx, NboundaryPadding);
    # reset a combined set of parameters (residual from old structure ... could be modified)
    # TODO: modify to pGrid, pModel, pInit
    p = (dx, mphi2, mchi2, c4, c, epsDiss, Nx, NboundaryPadding);

    # set the time span
    tspan = (0, target_time);

    # set the evolution method
    time_integration_method = RK4();
    
    # generate initial conditions
    u0 = initial_data(
        range(0, step=dx, length=(Nx + 2 * NboundaryPadding)),
        p, 
        pInit
    );
        
    # set the problem
    prob = ODEProblem(finite_differenced_pde_with_bc!, u0, tspan, p);

    sol = solve(
        prob, time_integration_method, 
        saveat = tspan[end]/10^3, #exp.(range(log(tspan[1]), log(tspan[end]), length=10^4))
        dt=dx/10, 
        adaptive = false, 
        dense=false, 
        maxiters=typemax(Int),
        callback=field_size_callback
    );
        
    # obtain the hamiltonian
    hamiltonian = zeros(length(sol.u))
    hamphi = zeros(length(sol.u))
    hamchi = zeros(length(sol.u))
    for i = 1:length(sol.u)
        hamiltonian[i] = nintegrate_simps(hamiltonian_density(sol.u[i], p), dx)
        hamphi[i] = nintegrate_simps(hamiltonian_phi(sol.u[i], p), dx)
        hamchi[i] = nintegrate_simps(hamiltonian_chi(sol.u[i], p), dx)
    end
    
    return (p, sol, hamiltonian, hamphi, hamchi)
end

artisan_evolution_at_resolution (generic function with 1 method)

In [4]:
function evolution_at_amplitude(amp, current_target_time, current_res_log2, stableRandomSeed)
    
    #stableRandomSeed = 42
    print("persistent random seed: ", stableRandomSeed, "\n")
    
    print("current characteristic amplitude: ", amp, "\n")
    
    # set monitoring flags
    convergence_maintained = false;
    lower_bound_only = true;
    
    # parameters of the model
    mphi2 = 1.;
    mchi2 = 15 * mphi2;
    c4 = 1.0;
    c = -1.;
    epsDiss = 0;

    # parameters of the initial data
    a0phi = amp; # effectively sets the relative amplitude to the stochastic ID (since Tkin is kept fixed)
    a0chi = a0phi; # effectively sets the relative amplitude to the stochastic ID (since Tkin is kept fixed)
    k0phi = 1;
    k0chi = 2 * k0phi;
    x0phi = 0;
    x0chi = 1/3;

    offsetphi = sqrt(((mphi2 - mchi2)^2 - 8 * mphi2 * c4) / (32 * c4^2))
    offsetchi = - sqrt(((mphi2 - mchi2)^2 - 8 * mchi2 * c4) / (32 * c4^2))

    aStochastic = 0;
    mink = 1;
    maxk = 4;
    
    desiredTkinPhi = NaN;
    desiredTkinChi = NaN;
    
    # set the combined set of parameters 
    pModel = (mphi2, mchi2, c4, c, epsDiss);
    pInit = (
        a0phi, a0chi, k0phi, k0chi, x0phi, x0chi, 
        offsetphi, offsetchi, 
        aStochastic, mink, maxk, stableRandomSeed,
        desiredTkinPhi, desiredTkinChi
    );
     
    # set some tables to store intermediate output
    resTab = [2^i for i in current_res_log2-2:current_res_log2]
    pTab = []
    solTab = []
    hamiltonianTab = []
    hamPhiTab = []
    hamChiTab = []
    
    #############################
    # evolution
    #############################
    
    # run evolution
    for res in resTab
        print("current resolution: ", res, "\n")
        # run the evolution
        @time (p, sol, hamiltonian, hamPhi, hamChi) = artisan_evolution_at_resolution(
            res, stableRandomSeed, pModel, pInit, current_target_time
        )
        print("... terminated", "\n")
        # unpack parameters
        (dx, mphi2, mchi2, c4, c, epsDiss, Nx, NboundaryPadding) = p
        # append the results
        push!(pTab, p)
        push!(solTab, sol)
        push!(hamiltonianTab, hamiltonian)
        push!(hamPhiTab, hamPhi)
        push!(hamChiTab, hamChi)
    end
    
    #############################
    # CONVERGENCE
    #############################
    
    dir_path = string("plots/",stableRandomSeed,"/",amp)

    # create the directory if it does not yet exist
    if !isdir(dir_path)
        print("Output plot directory does not exist. Creating it ...\n")
        mkpath(dir_path)
    else
        print("Output plot directory already exists.\n")
    end
    
    # plot and determine convergence 
    loss_of_convergence_time = save_convergence_plots(
        resTab, pTab, solTab, 
        hamiltonianTab, 
        dir_path
    )
    if loss_of_convergence_time >= solTab[end].t[end]
        print("Convergence kept at all times.\n")
    else
        print("Convergence lost at time t=",loss_of_convergence_time,"\n")
    end
    
    # determine the index of convergence loss
    loss_of_convergence_index = findfirst(t -> t > loss_of_convergence_time, solTab[end].t)
    if loss_of_convergence_index === nothing
        loss_of_convergence_index = length(solTab[end].t)
    end
    
    #print(10 * hamPhiTab[end][1],"\n")
    #print(hamPhiTab[end][2:10],"\n")
    
    # determine the onset time of the runaway (10-fold increase in either kinetic energy)
    runaway_index_phi = findfirst(
        energy -> abs(energy) > exp(1) * maximum(abs.(hamPhiTab[end][1:Int(div(length(hamPhiTab[end]),4))+1])), 
        hamPhiTab[end]
    )
    if runaway_index_phi === nothing
        runaway_index_phi = length(hamPhiTab[end])
    end
    runaway_index_chi = findfirst(
        energy -> abs(energy) > exp(1) * maximum(abs.(hamChiTab[end][1:Int(div(length(hamChiTab[end]),4))+1])), 
        hamChiTab[end]
    )
    if runaway_index_chi === nothing
        runaway_index_chi = length(hamChiTab[end])
    end
    runaway_index = min(runaway_index_phi, runaway_index_chi)
    runaway_time = solTab[end].t[runaway_index]
    if runaway_time >= solTab[end].t[end]
        print("No runaway detected.\n")
    else
        print("Runaway detected at time t=",runaway_time,"\n")
    end
    
    #############################
    # GENERATE REMAINING PLOTS IF DESIRED
    #############################
    
    dir_path = string("plots/",stableRandomSeed,"/",amp)

    # create the directory if it does not yet exist
    if !isdir(dir_path)
        #print("Directory does not exist. Creating it...")
        mkpath(dir_path)
    end
        
    # plot energy components
    save_energies_plot(
        resTab, pTab, solTab, 
        hamiltonianTab, hamPhiTab, hamChiTab, 
        dir_path,
        #loss_of_convergence_time=loss_of_convergence_time
    )
    save_normalised_energies_plot(
        resTab, pTab, solTab, 
        hamiltonianTab, hamPhiTab, hamChiTab, 
        dir_path,
        #loss_of_convergence_time=loss_of_convergence_time
    )
    save_difference_in_energies_plot(
        resTab, pTab, solTab, 
        hamiltonianTab, hamPhiTab, hamChiTab, 
        dir_path,
        #loss_of_convergence_time=loss_of_convergence_time
    )
    
    # plot field heatmaps
    save_density_plots(
        solTab[end], pTab[end], pInit,
        dir_path,
        loss_of_convergence_time=loss_of_convergence_time
    )
    
    # save snapshots
    save_snaps(
        solTab[end], pTab[end];
        snap_intervals=Int(round(length(solTab[end])/1)), 
        yrangeVal=1.2,
        dir_path = dir_path
    );
    
    # animate the fields
    save_animation(
        solTab[end][1:max(1,div(loss_of_convergence_index,10^2)):loss_of_convergence_index], 
        pTab[end],
        join([dir_path, "/animation_Nx=", resTab[end], ".gif"])
    );    
#     # animate frequencies
#     save_animation_momentum_space(
#         solTab[end][1:max(1,div(loss_of_convergence_index,10^2)):loss_of_convergence_index], 
#         pTab[end],
#         join([dir_path, "/animation_momentum_space_Nx=", resTab[end], ".gif"])
#     );
    
    print("Finished plotting.", "\n")
    
    #############################
    # SAVE DATA
    #############################
    
    if runaway_time > loss_of_convergence_time
        print("WARNING: Convergence not maintained until onset of runaway. Resolution insufficient.", "\n")
    else
        convergence_maintained = true
        if runaway_time >= solTab[end].t[end]
            print("WARNING: Lower bound only because target time insufficient.", "\n")
        else
            lower_bound_only = false
        end
    end
    
    dir_path = string("dat/",stableRandomSeed)
    if !isdir(dir_path)
        mkpath(dir_path)
    end

    timesteps = solTab[end].t
    stable_until = min(runaway_time, loss_of_convergence_time)

    @save joinpath(pwd(), dir_path, string(amp,".jld2")) amp stable_until lower_bound_only timesteps hamiltonianTab hamPhiTab hamChiTab

    print("Saved data.", "\n")

    
    return (runaway_time, convergence_maintained, lower_bound_only)
end

evolution_at_amplitude (generic function with 1 method)

### main()

In [5]:
function main()
    
    # some random seed (can be modified at will)
    stableRandomSeed = 0 #rand(1:10^7)
    
    # initialise flags
    convergence_maintained = false;
    lower_bound_only = true;
    
    # set abort criteria ...
    highest_res_log2 = 13;
    max_target_time = 2 * 10^4;
    # ... and their initial values
    current_res_log2 = 11;
    current_target_time = 10;
    
    # initialise the runaway time for handover to next amplitude
    runaway_time = Inf;
    
    # set table of desired amplitudes (NOTE: links to scaling assumption below)
    amp_base = 1
    amplitudes = [invAmp for invAmp in 0.5:0.5:16]
    amplitudes = amplitudes.^(-1)
    
    # loop over all amplitudes
    for amp in amplitudes
        
        # re-attempt while flags not positive or until abort criteria met
        while (!convergence_maintained||lower_bound_only) && (current_res_log2 <= highest_res_log2) && (current_target_time <= max_target_time)
            # attempt run and obtain flags
            (runaway_time, convergence_maintained, lower_bound_only) = evolution_at_amplitude(
                amp, 
                current_target_time, 
                current_res_log2,
                stableRandomSeed
            )
            # update according to obtained flags
            if convergence_maintained                
                if lower_bound_only
                    current_target_time = current_target_time * 4
                    print("Increasing target time to T = ", current_target_time, "\n")
                else
                    current_target_time = min(runaway_time, current_target_time);
                    print("Target time reset to confidently detected runaway time T = ", current_target_time, "\n")
                end
            else
                current_res_log2 = current_res_log2 + 1;
                print("Increasing resolution from N = ", current_res_log2 - 1, " to ", current_res_log2, "\n")
            end
        end
        
        print("AMPLITUDE A = ", amp, " DONE!\n")
        
        # update target time based on the presumed scaling assumption and adapt the target time accordingly
        current_target_time = runaway_time
        print("Updating target time for next amplitude from T = ", current_target_time, " ... ")
        current_target_time = current_target_time * exp(amp_base)     
        print("to T = ", current_target_time, "\n")
        
        # decrease resolution if convergence was maintained in previous step
#         if convergence_maintained
#             current_res_log2 = current_res_log2 - 1;
#             print("Decreasing resolution from N = ", current_res_log2 + 1, " to ", current_res_log2, "\n")
#         end
        
        # check whether it makes sense to go on; otherwise abort 
        if convergence_maintained && lower_bound_only && current_target_time >= max_target_time
            print("ABORT: maximum target time approached in converged simulation; no use to proceed")
            return
        end
        
        # ensure that current params don't exceed the abort criteria for the next step
        current_res_log2 = min(current_res_log2, highest_res_log2)
        current_target_time = min(current_target_time, max_target_time)
        
        # reset the flags
        convergence_maintained = false;
        lower_bound_only = true;
    end

end

main (generic function with 1 method)

In [ ]:
main()

persistent random seed: 0
current characteristic amplitude: 2.0
current resolution: 256
	user-assigned no rescaling:
		max |amplitude| phi before rescaling: 2.0
		max |amplitude| chi before rescaling: 1.9997322758191236
Terminating because one of the fields grew too large at time t = 3.1054687500004383.
  4.127406 seconds (3.78 M allocations: 1.031 GiB, 4.18% gc time, 88.99% compilation time)
... terminated
current resolution: 512
	user-assigned no rescaling:
		max |amplitude| phi before rescaling: 2.0
		max |amplitude| chi before rescaling: 1.9999330678348022
Terminating because one of the fields grew too large at time t = 3.2283203124992035.
  1.360494 seconds (1.20 M allocations: 3.214 GiB, 7.97% gc time)
... terminated
current resolution: 1024
	user-assigned no rescaling:
		max |amplitude| phi before rescaling: 2.0
		max |amplitude| chi before rescaling: 1.9999832668887012
Terminating because one of the fields grew too large at time t = 3.313769531248056.
  4.821007 seconds (2.46 M

[ Info: Saved animation to /home/aaron/Dropbox/university/0_research/2022-2025/stable_ghosts_Paris/field_theory/calculations_aaron/julia_code/1+1/v6_on_Thanos/02_Liouville/02_nontrivial_vauum/03_longlived_scalings/L=1_C4=1_m2=1/01_amplitude/02_wave/plots/0/2.0/animation_Nx=1024.gif


Saved data.
Target time reset to confidently detected runaway time T = 1.15
AMPLITUDE A = 2.0 DONE!
Updating target time for next amplitude from T = 1.15 ... to T = 3.1260241027279014
persistent random seed: 0
current characteristic amplitude: 1.0
current resolution: 256
	user-assigned no rescaling:
		max |amplitude| phi before rescaling: 1.0
		max |amplitude| chi before rescaling: 0.9998661379095618
  0.594358 seconds (666.25 k allocations: 885.081 MiB, 10.71% gc time, 13.55% compilation time: 13% of which was recompilation)
... terminated
current resolution: 512
	user-assigned no rescaling:
		max |amplitude| phi before rescaling: 1.0
		max |amplitude| chi before rescaling: 0.9999665339174011
  2.415810 seconds (1.19 M allocations: 3.175 GiB, 43.18% gc time)
... terminated
current resolution: 1024
	user-assigned no rescaling:
		max |amplitude| phi before rescaling: 1.0
		max |amplitude| chi before rescaling: 0.9999916334443506
  4.727519 seconds (2.35 M allocations: 12.206 GiB, 6.85% 

[ Info: Saved animation to /home/aaron/Dropbox/university/0_research/2022-2025/stable_ghosts_Paris/field_theory/calculations_aaron/julia_code/1+1/v6_on_Thanos/02_Liouville/02_nontrivial_vauum/03_longlived_scalings/L=1_C4=1_m2=1/01_amplitude/02_wave/plots/0/1.0/animation_Nx=1024.gif


  0.801806 seconds (961.73 k allocations: 1.419 GiB, 8.05% gc time)
... terminated
current resolution: 512
	user-assigned no rescaling:
		max |amplitude| phi before rescaling: 0.6666666666666666
		max |amplitude| chi before rescaling: 0.666644355944934
  2.400196 seconds (1.99 M allocations: 5.305 GiB, 8.19% gc time)
... terminated
current resolution: 1024
	user-assigned no rescaling:
		max |amplitude| phi before rescaling: 0.6666666666666666
		max |amplitude| chi before rescaling: 0.6666610889629003
  7.811720 seconds (3.94 M allocations: 20.514 GiB, 6.78% gc time)
... terminated
Output plot directory does not exist. Creating it ...
Convergence kept at all times.
Runaway detected at time t=3.9746146543289504
Finished plotting.
Saved data.
Target time reset to confidently detected runaway time T = 3.9746146543289504
AMPLITUDE A = 0.6666666666666666 DONE!
Updating target time for next amplitude from T = 3.9746146543289504 ... to T = 10.804122789989416
persistent random seed: 0
current c

...

Terminating because one of the fields grew too large at time t = 248.02089844317132.
141.682204 seconds (91.45 M allocations: 244.671 GiB, 9.52% gc time)
... terminated
current resolution: 1024
	user-assigned no rescaling:
		max |amplitude| phi before rescaling: 0.14285714285714285
		max |amplitude| chi before rescaling: 0.14285594763490722
Terminating because one of the fields grew too large at time t = 246.37070313495795.
493.940782 seconds (181.67 M allocations: 948.030 GiB, 9.21% gc time)
... terminated
current resolution: 2048
	user-assigned no rescaling:
		max |amplitude| phi before rescaling: 0.14285714285714285
		max |amplitude| chi before rescaling: 0.14285684405127144
Terminating because one of the fields grew too large at time t = 246.87343747758183.
1900.615900 seconds (525.88 M allocations: 3.652 TiB, 9.20% gc time)
... terminated
Output plot directory does not exist. Creating it ...
Convergence lost at time t=237.34079974353637
Runaway detected at time t=175.9309424672367

[ Info: Saved animation to /home/aaron/Dropbox/university/0_research/2022-2025/stable_ghosts_Paris/field_theory/calculations_aaron/julia_code/1+1/v6_on_Thanos/02_Liouville/02_nontrivial_vauum/03_longlived_scalings/L=1_C4=1_m2=1/01_amplitude/02_wave/plots/0/0.14285714285714285/animation_Nx=2048.gif


Terminating because one of the fields grew too large at time t = 261.58750000646097.
156.769081 seconds (96.46 M allocations: 258.061 GiB, 9.59% gc time)
... terminated
current resolution: 1024
	user-assigned no rescaling:
		max |amplitude| phi before rescaling: 0.13333333333333333
		max |amplitude| chi before rescaling: 0.1333322177925801
Terminating because one of the fields grew too large at time t = 261.2644531342927.
544.693671 seconds (192.66 M allocations: 1005.354 GiB, 9.50% gc time)
... terminated
current resolution: 2048
	user-assigned no rescaling:
		max |amplitude| phi before rescaling: 0.13333333333333333
		max |amplitude| chi before rescaling: 0.13333305444785337
Terminating because one of the fields grew too large at time t = 250.18105466431172.
2042.449695 seconds (532.94 M allocations: 3.701 TiB, 9.13% gc time)
... terminated
Output plot directory does not exist. Creating it ...
Convergence lost at time t=240.5496316380988
Runaway detected at time t=221.8986661631766
F

[ Info: Saved animation to /home/aaron/Dropbox/university/0_research/2022-2025/stable_ghosts_Paris/field_theory/calculations_aaron/julia_code/1+1/v6_on_Thanos/02_Liouville/02_nontrivial_vauum/03_longlived_scalings/L=1_C4=1_m2=1/01_amplitude/02_wave/plots/0/0.13333333333333333/animation_Nx=2048.gif


Terminating because one of the fields grew too large at time t = 290.1595703206241.
191.892783 seconds (106.99 M allocations: 286.237 GiB, 9.37% gc time)
... terminated
current resolution: 1024
	user-assigned no rescaling:
		max |amplitude| phi before rescaling: 0.125
		max |amplitude| chi before rescaling: 0.12499895418054383
Terminating because one of the fields grew too large at time t = 287.9641601593262.
698.033534 seconds (212.34 M allocations: 1.082 TiB, 8.97% gc time)
... terminated
current resolution: 2048
	user-assigned no rescaling:
		max |amplitude| phi before rescaling: 0.125
		max |amplitude| chi before rescaling: 0.12499973854486253


### export .jl for production run

In [2]:
using NBInclude
nbexport("main.jl", "main.ipynb")